# Phone Detection Training - YOLO11 on Kaggle T4x2

Notebook này dùng để train model phát hiện điện thoại cho dự án **AI Driver Safety Assessment System**.

**Thiết lập Kaggle khuyến nghị:**
- Accelerator: GPU T4 x2
- Internet: bật ở lần chạy đầu để cài `ultralytics` nếu Kaggle image chưa có
- Dataset: upload file `mobilephone_wusj2_v7_yolo11_kaggle.zip` dưới dạng Kaggle Dataset
- Output chính: `best.pt`, `last.pt`, metrics và file zip tổng hợp để tải về


In [ ]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

def ensure_package(package_name, import_name=None):
    import importlib.util
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
    else:
        print(f'{package_name} is already available')

ensure_package('ultralytics')

import torch
import yaml
from ultralytics import YOLO

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Tìm dataset và chuẩn hóa `data.yaml`

Cell này tự tìm `data.yaml` trong `/kaggle/input`, sửa lại đường dẫn train/valid/test nếu cần, và ghi bản YAML làm việc vào `/kaggle/working/data.yaml`.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
RUNS_DIR = WORKING_DIR / 'runs'
RUN_NAME = 'phone_yolo11_t4x2'

yaml_candidates = sorted(INPUT_ROOT.rglob('data.yaml'))
if not yaml_candidates:
    zip_candidates = sorted(INPUT_ROOT.rglob('*.zip'))
    if zip_candidates:
        extract_root = WORKING_DIR / 'extracted_dataset'
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        print('No extracted data.yaml found. Extracting zip:', zip_candidates[0])
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(extract_root)
        yaml_candidates = sorted(extract_root.rglob('data.yaml'))
if not yaml_candidates:
    raise FileNotFoundError('Không tìm thấy data.yaml trong /kaggle/input hoặc trong file .zip dataset.')

source_yaml = yaml_candidates[0]
dataset_root = source_yaml.parent
print('Using dataset YAML:', source_yaml)
print('Dataset root:', dataset_root)

with open(source_yaml, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = str(dataset_root)
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
data_cfg['test'] = 'test/images'
data_cfg['nc'] = 1
data_cfg['names'] = ['Mobile-phone']

work_yaml = WORKING_DIR / 'mobilephone_data.yaml'
with open(work_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print(work_yaml.read_text())


## 2. Kiểm tra dataset trước khi train

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_split(split):
    image_dir = dataset_root / split / 'images'
    label_dir = dataset_root / split / 'labels'
    images = [p for p in image_dir.glob('*') if p.suffix.lower() in IMG_EXTS]
    labels = list(label_dir.glob('*.txt')) if label_dir.exists() else []
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    boxes = 0
    bad = 0
    for label in labels:
        text = label.read_text(encoding='utf-8', errors='replace').strip()
        for line in text.splitlines():
            parts = line.split()
            if len(parts) < 5:
                bad += 1
                continue
            boxes += 1
            try:
                vals = list(map(float, parts[1:5]))
                if any(v < 0 or v > 1 for v in vals) or vals[2] <= 0 or vals[3] <= 0:
                    bad += 1
            except Exception:
                bad += 1
    return {
        'images': len(images),
        'labels': len(labels),
        'missing_labels': len(image_stems - label_stems),
        'extra_labels': len(label_stems - image_stems),
        'boxes': boxes,
        'bad_label_lines': bad,
    }

stats = {split: count_split(split) for split in ['train', 'valid', 'test']}
stats


In [ ]:
for split, row in stats.items():
    assert row['images'] > 0, f'{split} không có ảnh'
    assert row['labels'] > 0, f'{split} không có label'
    assert row['missing_labels'] == 0, f'{split} thiếu label cho ảnh'
    assert row['bad_label_lines'] == 0, f'{split} có dòng label lỗi'
print('Dataset sanity check passed.')


## 3. Train YOLO11

Mặc định dùng `yolo11n.pt` để train nhanh và ổn định. Nếu muốn mạnh hơn, đổi `MODEL_NAME` thành `yolo11s.pt` hoặc `yolo11m.pt`, nhưng thời gian train và VRAM sẽ tăng.

In [ ]:
MODEL_NAME = 'yolo11n.pt'
EPOCHS = 80
IMGSZ = 640
BATCH = 32
PATIENCE = 20

if torch.cuda.device_count() >= 2:
    DEVICE = '0,1'
elif torch.cuda.device_count() == 1:
    DEVICE = 0
else:
    DEVICE = 'cpu'

print('Training device:', DEVICE)

def run_train(device, batch):
    model = YOLO(MODEL_NAME)
    return model.train(
        data=str(work_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=batch,
        device=device,
        workers=2,
        patience=PATIENCE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        optimizer='auto',
        cos_lr=True,
        amp=True,
        cache=False,
        plots=True,
    )

try:
    results = run_train(DEVICE, BATCH)
except Exception as exc:
    if DEVICE == '0,1':
        print('Multi-GPU training failed, falling back to single GPU. Error:', exc)
        DEVICE = 0
        BATCH = min(BATCH, 16)
        results = run_train(DEVICE, BATCH)
    else:
        raise


## 4. Validate trên tập test

In [ ]:
best_pt = RUNS_DIR / 'detect' / RUN_NAME / 'weights' / 'best.pt'
last_pt = RUNS_DIR / 'detect' / RUN_NAME / 'weights' / 'last.pt'
assert best_pt.exists(), f'Không tìm thấy best.pt tại {best_pt}'

best_model = YOLO(str(best_pt))
metrics = best_model.val(
    data=str(work_yaml),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME + '_test_eval',
    exist_ok=True,
    plots=True,
)
print(metrics)


## 5. Chạy thử predict và đóng gói output

In [ ]:
predict_dir = RUNS_DIR / 'predict_samples'
sample_images = sorted((dataset_root / 'test' / 'images').glob('*'))[:24]
assert sample_images, 'Không tìm thấy ảnh test để predict mẫu'

best_model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(RUNS_DIR),
    name='predict_samples',
    exist_ok=True,
    save=True,
)

package_dir = WORKING_DIR / 'phone_detection_export'
package_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, package_dir / 'phone_yolo11_best.pt')
if last_pt.exists():
    shutil.copy2(last_pt, package_dir / 'phone_yolo11_last.pt')
shutil.copy2(work_yaml, package_dir / 'mobilephone_data.yaml')

run_dir = RUNS_DIR / 'detect' / RUN_NAME
for filename in ['results.csv', 'results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png']:
    src = run_dir / filename
    if src.exists():
        shutil.copy2(src, package_dir / filename)

zip_base = WORKING_DIR / 'phone_detection_training_results'
zip_path = shutil.make_archive(str(zip_base), 'zip', package_dir)
print('Export folder:', package_dir)
print('Download zip:', zip_path)


## 6. Cách dùng model trong dự án

Sau khi tải `phone_detection_training_results.zip`, lấy file `phone_yolo11_best.pt` và đặt vào:

`D:\\Work\\HocKy3\\ComputerVision\\MDDAD\\models\\phone_yolo11n.pt`

Sau đó vào trang `/phone-test`, chọn **YOLO fine-tuned** hoặc nhập custom model path để test bounding box.